# 11. 파이썬 기초 - Streamlit 웹 앱

`streamlit` 은 **파이썬 코드만으로 웹 앱** 을 만드는 라이브러리입니다.
HTML·CSS·JavaScript 없이 분석 결과를 바로 웹으로 공유할 수 있습니다.

이 프로젝트의 `streamlit_examples/` 폴더에 14개 예제가 있는데,
그 예제들을 이해하려면 **먼저 알아야 하는 원리** 를 다룹니다.

**다루는 내용**
1. 실행 모델 — Streamlit 의 가장 중요한 개념 ⭐
2. 기본 출력
3. 위젯과 입력값
4. `session_state` — 재실행돼도 값을 유지
5. `cache_data` — 매번 다시 읽지 않게
6. 레이아웃 (sidebar / columns / tabs)
7. 차트 그리기
8. 실행해 보기

> **주의**: `st.*` 코드는 주피터에서 실행되지 않습니다.
> `streamlit run 파일.py` 로만 동작하므로, 이 노트북은
> `%%writefile` 로 **실행 가능한 .py 파일을 만들어 두는 방식** 으로 진행합니다.

In [1]:
import streamlit as st
import os

print('streamlit', st.__version__)

# 이 노트북에서 만든 예제를 모아둘 폴더
os.makedirs('streamlit_basic', exist_ok=True)
print('예제 저장 폴더: python_basic/streamlit_basic/')

streamlit 1.37.1
예제 저장 폴더: python_basic/streamlit_basic/


## 1. 실행 모델 — 가장 중요한 개념 ⭐

Streamlit 을 처음 쓸 때 가장 많이 혼란스러워하는 부분입니다.

> **사용자가 위젯을 건드릴 때마다, 스크립트가 처음부터 끝까지 전부 다시 실행된다.**

일반 웹 프레임워크처럼 "이 버튼을 누르면 이 함수만 실행" 이 아닙니다.
버튼 하나만 눌러도 `import` 부터 마지막 줄까지 **전체가 재실행** 됩니다.

```
사용자가 슬라이더를 움직임
        ↓
스크립트 1번 줄부터 끝까지 재실행
        ↓
화면 전체를 새로 그림
```

**여기서 두 가지 문제가 생깁니다.**

| 문제 | 해결책 |
|---|---|
| 재실행되면 변수가 초기화된다 → 값이 사라짐 | `st.session_state` (4번) |
| 재실행마다 CSV·API를 다시 읽는다 → 느림 | `st.cache_data` (5번) |

이 두 가지가 예제 파일에서 각각 57회·7회 쓰인 이유입니다.

In [2]:
# 재실행이 무슨 뜻인지 파이썬으로 흉내내 보기
#   아래 count 는 함수가 호출될 때마다 0 으로 초기화된다.
#   Streamlit 의 스크립트 전체가 딱 이렇게 동작한다.

def streamlit_script_흉내():
    """위젯 조작 1회 = 이 함수 1회 호출 이라고 생각하면 된다."""
    count = 0          # 매번 0 으로 초기화됨!
    count = count + 1
    return count


print('1번째 조작:', streamlit_script_흉내())
print('2번째 조작:', streamlit_script_흉내())
print('3번째 조작:', streamlit_script_흉내())
print()
print('→ 아무리 눌러도 1. 값이 누적되지 않는다.')
print('→ 이 문제를 푸는 것이 session_state 다.')

1번째 조작: 1
2번째 조작: 1
3번째 조작: 1

→ 아무리 눌러도 1. 값이 누적되지 않는다.
→ 이 문제를 푸는 것이 session_state 다.


## 2. 기본 출력

| 함수 | 용도 |
|---|---|
| `st.title` / `st.header` / `st.subheader` | 제목 |
| `st.write` | **아무거나** 알아서 출력 (문자열·표·그래프) |
| `st.markdown` | 마크다운 |
| `st.dataframe` | 표 (정렬·검색 가능) |
| `st.metric` | 숫자 카드 |
| `st.info` / `st.success` / `st.warning` / `st.error` | 색깔 있는 알림 상자 |

In [3]:
%%writefile streamlit_basic/01_기본출력.py
# 실행: streamlit run streamlit_basic/01_기본출력.py
import streamlit as st
import pandas as pd

# set_page_config 는 반드시 '가장 먼저' 호출해야 한다
st.set_page_config(page_title='기본 출력', layout='wide')

st.title('01. 기본 출력')
st.header('헤더')
st.subheader('서브헤더')

# write 는 만능이다 - 문자열, 숫자, DataFrame, 그래프 모두 알아서 처리
st.write('일반 텍스트입니다.')
st.write({'딕셔너리도': '표로 보여준다', 'key': 'value'})

st.divider()   # 가로 구분선

df = pd.DataFrame({
    '이름': ['지민', '수진', '태양'],
    '점수': [90, 85, 78],
})

st.subheader('표 출력')
st.dataframe(df, use_container_width=True)   # 정렬·검색 가능한 표
st.table(df)                                  # 정적인 표

st.divider()

# metric: 대시보드의 숫자 카드
col1, col2 = st.columns(2)
col1.metric('평균 점수', f"{df['점수'].mean():.1f}", delta='2.3')
col2.metric('최고 점수', df['점수'].max())

st.divider()

# 알림 상자 4종
st.info('정보 메시지')
st.success('성공 메시지')
st.warning('경고 메시지')
st.error('에러 메시지')

Overwriting streamlit_basic/01_기본출력.py


## 3. 위젯과 입력값

위젯 함수는 **사용자가 선택한 값을 그대로 반환** 합니다.
콜백 함수를 등록하는 게 아니라, **반환값을 변수로 받아 쓰는** 방식입니다.

```python
name = st.text_input('이름')     # name 에 입력값이 들어있다
if st.button('확인'):            # 눌렸으면 True
    st.write(f'안녕하세요 {name}님')
```

> `st.button` 은 **눌린 그 순간의 재실행에서만 True** 이고, 다음 재실행에서는 다시 False 입니다.
> 이것이 4번 `session_state` 가 필요한 대표적인 이유입니다.

In [4]:
%%writefile streamlit_basic/02_위젯.py
# 실행: streamlit run streamlit_basic/02_위젯.py
import streamlit as st

st.title('02. 위젯 - 입력값 받기')

# 각 위젯은 '사용자가 고른 값' 을 반환한다
name = st.text_input('이름을 입력하세요', value='홍길동')
age = st.slider('나이', min_value=0, max_value=100, value=25)
city = st.selectbox('도시', ['서울', '부산', '대구'])
hobbies = st.multiselect('취미', ['독서', '운동', '게임', '음악'])
agree = st.checkbox('약관에 동의합니다')
gender = st.radio('성별', ['남', '여'], horizontal=True)

st.divider()

# 위젯을 하나라도 건드리면 스크립트가 처음부터 재실행되고
# 아래 내용이 새 값으로 다시 그려진다
st.subheader('입력 결과')
st.write(f'**{name}** / {age}세 / {city} / {gender}')
st.write('취미:', ', '.join(hobbies) if hobbies else '(없음)')

if not agree:
    st.warning('약관에 동의해야 제출할 수 있습니다.')
elif st.button('제출'):
    # button 은 눌린 그 순간의 재실행에서만 True 다
    st.success(f'{name}님 제출 완료!')
    st.balloons()

Overwriting streamlit_basic/02_위젯.py


## 4. `session_state` — 재실행돼도 값을 유지 ⭐

1번에서 본 문제(재실행 시 변수 초기화)를 푸는 장치입니다.

`st.session_state` 는 **재실행을 넘어 살아남는 딕셔너리** 입니다.

```python
# 최초 1회만 초기화 (이 if 가 없으면 매번 0 으로 덮어써진다)
if 'count' not in st.session_state:
    st.session_state.count = 0

if st.button('증가'):
    st.session_state.count += 1      # 이 값은 다음 재실행에도 남아있다

st.write(st.session_state.count)
```

**`if '키' not in st.session_state:` 로 감싸는 것이 핵심 패턴** 입니다.
예제 파일들에서 검색 결과·로그인 상태·장바구니를 이 방식으로 보관합니다.

In [5]:
%%writefile streamlit_basic/03_session_state.py
# 실행: streamlit run streamlit_basic/03_session_state.py
import streamlit as st

st.title('03. session_state - 값 유지하기')

# ---------------------------------------------------------
# 잘못된 예: 매 재실행마다 0 으로 초기화되어 절대 늘지 않는다
# ---------------------------------------------------------
bad_count = 0
if st.button('❌ 잘못된 카운터'):
    bad_count += 1
st.write('잘못된 카운터:', bad_count, '← 아무리 눌러도 0 또는 1')

st.divider()

# ---------------------------------------------------------
# 올바른 예: session_state 에 보관하면 재실행돼도 살아남는다
# ---------------------------------------------------------
# 이 if 문이 '최초 1회만 초기화' 를 보장한다
if 'count' not in st.session_state:
    st.session_state.count = 0

col1, col2 = st.columns(2)
if col1.button('⭕ 올바른 카운터 +1'):
    st.session_state.count += 1
if col2.button('초기화'):
    st.session_state.count = 0

st.write('올바른 카운터:', st.session_state.count, '← 계속 누적된다')

st.divider()

# 실전 패턴: 검색 결과를 보관해 두면
# 다른 위젯을 건드려도 결과가 사라지지 않는다
if 'history' not in st.session_state:
    st.session_state.history = []

keyword = st.text_input('검색어')
if st.button('검색') and keyword:
    st.session_state.history.append(keyword)

st.subheader('검색 기록')
st.write(st.session_state.history)

# 디버깅용: 현재 보관 중인 전체 상태 확인
with st.expander('session_state 전체 보기'):
    st.write(dict(st.session_state))

Overwriting streamlit_basic/03_session_state.py


## 5. `cache_data` — 매번 다시 읽지 않게 ⭐

재실행될 때마다 **CSV를 다시 읽고 API를 다시 호출하면** 앱이 매우 느려집니다.

`@st.cache_data` 를 함수에 붙이면 **같은 인자로 호출될 때 저장해둔 결과를 재사용** 합니다.

```python
@st.cache_data
def load_data():
    return pd.read_csv('../data/netflix_titles.csv')   # 최초 1회만 실제 실행

df = load_data()   # 두 번째부터는 즉시 반환
```

- 데이터(표·값) → `@st.cache_data`
- 모델·DB연결 같은 무거운 객체 → `@st.cache_resource`

In [6]:
%%writefile streamlit_basic/04_cache.py
# 실행: streamlit run streamlit_basic/04_cache.py
import streamlit as st
import pandas as pd
import time
from pathlib import Path

st.title('04. cache_data - 다시 읽지 않기')

# streamlit run 은 '명령을 실행한 폴더' 가 기준(cwd)이 된다. 스크립트 위치가 아니다.
# 따라서 './../data' 같은 상대경로는 어디서 실행하느냐에 따라 깨진다.
# __file__(이 스크립트의 위치) 기준으로 절대경로를 만들면 어디서 실행해도 안전하다.
#   streamlit_basic/04_cache.py → parents[0]=streamlit_basic
#                                 parents[1]=python_basic
#                                 parents[2]=프로젝트 루트
CSV_PATH = Path(__file__).resolve().parents[2] / 'data' / 'netflix_titles.csv'


@st.cache_data
def load_data(path):
    """CSV 를 읽어 DataFrame 으로 돌려준다.

    @st.cache_data 덕분에 같은 path 로 호출되면
    실제 읽기는 최초 1회만 수행되고, 이후에는 저장된 결과를 반환한다.
    """
    time.sleep(2)               # 느린 작업을 흉내
    return pd.read_csv(path)


start = time.time()
df = load_data(CSV_PATH)
elapsed = time.time() - start

st.metric('로딩 시간', f'{elapsed:.2f} 초')
st.info('처음에는 2초 이상, 이후 재실행에서는 0초에 가깝게 나온다.')

st.dataframe(df.head(20), use_container_width=True)

# 슬라이더를 움직여 재실행시켜 보자 - 로딩 시간이 0초가 된다
n = st.slider('표시할 행 수', 5, 50, 20)
st.write(f'상위 {n}개 행')
st.dataframe(df.head(n), use_container_width=True)

if st.button('캐시 지우기'):
    st.cache_data.clear()
    st.rerun()      # 스크립트를 강제로 다시 실행

Overwriting streamlit_basic/04_cache.py


## 6. 레이아웃 — sidebar / columns / tabs

| 문법 | 결과 |
|---|---|
| `st.sidebar.xxx` 또는 `with st.sidebar:` | 왼쪽 사이드바에 배치 |
| `c1, c2 = st.columns(2)` | 좌우로 나눠 배치 |
| `t1, t2 = st.tabs(['A','B'])` | 탭으로 전환 |
| `with st.expander('제목'):` | 접었다 펼치는 영역 |

**필터는 사이드바, 결과는 본문** 이 가장 흔한 구성입니다. (예제 파일에서 sidebar 48회 사용)

In [7]:
%%writefile streamlit_basic/05_레이아웃.py
# 실행: streamlit run streamlit_basic/05_레이아웃.py
import streamlit as st
import pandas as pd
from pathlib import Path

st.set_page_config(page_title='레이아웃', layout='wide')
st.title('05. 레이아웃')

# 실행 위치와 무관하게 동작하도록 __file__ 기준 절대경로 사용
CSV_PATH = Path(__file__).resolve().parents[2] / 'data' / 'netflix_titles.csv'


@st.cache_data
def load_data():
    return pd.read_csv(CSV_PATH)


df = load_data()

# ---------- 사이드바: 필터 ----------
with st.sidebar:
    st.header('🔍 필터')
    kind = st.selectbox('종류', ['전체'] + df['type'].dropna().unique().tolist())
    year_min, year_max = st.slider(
        '제작 연도',
        int(df['release_year'].min()), int(df['release_year'].max()),
        (2015, 2021),
    )

# ---------- 본문: 필터 적용 결과 ----------
filtered = df[df['release_year'].between(year_min, year_max)]
if kind != '전체':
    filtered = filtered[filtered['type'] == kind]

# columns: 숫자 카드를 가로로 배치
c1, c2, c3 = st.columns(3)
c1.metric('전체 작품', f'{len(df):,}')
c2.metric('필터 결과', f'{len(filtered):,}')
c3.metric('비율', f'{len(filtered) / len(df) * 100:.1f}%')

st.divider()

# tabs: 같은 데이터를 여러 관점으로
tab1, tab2, tab3 = st.tabs(['📋 목록', '📊 연도별', '🎬 장르'])

with tab1:
    st.dataframe(
        filtered[['type', 'title', 'release_year', 'listed_in']].head(100),
        use_container_width=True,
    )

with tab2:
    by_year = filtered['release_year'].value_counts().sort_index()
    st.bar_chart(by_year)

with tab3:
    # 08편에서 배운 split → explode
    genres = filtered['listed_in'].str.split(', ').explode()
    st.bar_chart(genres.value_counts().head(10))

with st.expander('원본 데이터 정보'):
    st.write('행 x 열:', df.shape)
    st.write('컬럼:', list(df.columns))

Overwriting streamlit_basic/05_레이아웃.py


## 7. 차트 그리기

| 방법 | 특징 |
|---|---|
| `st.bar_chart(데이터)` | 가장 간단. Series/DataFrame 을 그대로 |
| `st.line_chart` / `st.area_chart` | 선/영역 그래프 |
| `st.pyplot(fig)` | **matplotlib Figure** 를 그대로 (06편 내용 재사용) |

`st.pyplot` 을 쓰면 06편에서 배운 matplotlib 코드를 **그대로 웹에 올릴 수 있습니다.**
단, 한글 폰트 설정은 여기서도 필요합니다.

In [8]:
%%writefile streamlit_basic/06_차트.py
# 실행: streamlit run streamlit_basic/06_차트.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rc
from pathlib import Path

# 한글 폰트 설정 (06편과 동일)
rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

st.title('06. 차트')

# 실행 위치와 무관하게 동작하도록 __file__ 기준 절대경로 사용
CSV_PATH = Path(__file__).resolve().parents[2] / 'data' / 'netflix_titles.csv'


@st.cache_data
def load_data():
    return pd.read_csv(CSV_PATH)


df = load_data()
by_year = df[df['release_year'] >= 2010]['release_year'].value_counts().sort_index()

# ---------- 방법 1: streamlit 내장 차트 (간단) ----------
st.subheader('내장 차트 - st.bar_chart')
st.bar_chart(by_year)

st.divider()

# ---------- 방법 2: matplotlib (세밀한 제어 가능) ----------
st.subheader('matplotlib - st.pyplot')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_year.index, by_year.values, color='#E50914')
ax.set_title('연도별 넷플릭스 콘텐츠 수')
ax.set_xlabel('제작 연도')
ax.set_ylabel('작품 수')
ax.grid(axis='y', alpha=0.3)

st.pyplot(fig)      # Figure 객체를 그대로 넘긴다

st.caption('matplotlib 은 제목·색·격자 등을 세밀하게 제어할 수 있다.')

Overwriting streamlit_basic/06_차트.py


## 8. 실행해 보기

터미널(PowerShell)에서 아래 명령으로 실행합니다.

```powershell
cd C:\myclass\3python\python_web\Python_Webscraping_Analysis-rookies5\python_basic
streamlit run streamlit_basic/01_기본출력.py
```

- 브라우저가 자동으로 열립니다 (기본 주소 `http://localhost:8501`)
- **코드를 저장하면 브라우저 우측 상단에 "Rerun" 버튼** 이 뜹니다
- 종료는 터미널에서 `Ctrl + C`

> 주피터 셀에서 `!streamlit run ...` 으로 실행하면 서버가 계속 떠 있어
> 셀이 끝나지 않습니다. **반드시 터미널에서 실행** 하세요.

In [9]:
# 생성된 예제 파일 확인
import os

files = sorted(os.listdir('streamlit_basic'))
print('python_basic/streamlit_basic/ 에 생성된 예제:')
for f in files:
    size = os.path.getsize(os.path.join('streamlit_basic', f))
    print(f'  {f:<24} {size:>5} bytes')

print()
print('실행 명령 예시:')
print('  streamlit run streamlit_basic/03_session_state.py')

python_basic/streamlit_basic/ 에 생성된 예제:
  01_기본출력.py                1210 bytes
  02_위젯.py                  1129 bytes
  03_session_state.py       1669 bytes
  04_cache.py               1764 bytes
  05_레이아웃.py                1962 bytes
  06_차트.py                  1351 bytes

실행 명령 예시:
  streamlit run streamlit_basic/03_session_state.py


## 정리

### 가장 중요한 것: 실행 모델
> **위젯을 건드릴 때마다 스크립트가 처음부터 끝까지 다시 실행된다.**

이 하나에서 나머지가 전부 파생됩니다.

| 문제 | 해결 | 패턴 |
|---|---|---|
| 재실행 시 변수 초기화 | `st.session_state` | `if '키' not in st.session_state:` 로 최초 1회만 초기화 |
| 재실행마다 데이터 재로딩 | `@st.cache_data` | 데이터를 읽는 함수 위에 붙이기 |

### 자주 쓰는 문법

```python
st.set_page_config(page_title='제목', layout='wide')   # 맨 위에서 1회
st.title / header / subheader / write / markdown        # 출력
st.dataframe(df, use_container_width=True)              # 표
st.metric('라벨', 값, delta=변화량)                       # 숫자 카드
값 = st.text_input / slider / selectbox / multiselect    # 입력 (반환값을 받는다)
with st.sidebar:  ...                                    # 사이드바
c1, c2 = st.columns(2)                                   # 좌우 분할
t1, t2 = st.tabs(['A', 'B'])                             # 탭
st.bar_chart(series)  /  st.pyplot(fig)                  # 차트
```

### 다음 단계
`streamlit_examples/` 폴더의 예제를 이 순서로 읽으면 이해하기 쉽습니다.

1. `00streamlit_basic.py` — 최소 구성
2. `02streamlit_uicomponent.py` — 위젯 모음
3. `04streamlit_korea.py` → `06streamlit_korea_func.py` — 데이터 + 필터
4. `08streamlit_blog_search_nhncloud.py` — API 연동 + `session_state`
5. `11streamlit_netflix_dashboard.py` — 대시보드 종합